# LLM-Based Free-Response Tasks Analysis

This notebook contains Python code for reproducing the results in our paper:

Van Campenhout, R., Dittel, J. S., Jerome, B., & Johnson, B. G. (2026). Extending an automatic question generation pipeline with LLM-based free-response tasks: An analysis of performance metrics using student data. In *Proceedings of the 18th International Conference on Computer Supported Education* (Vol. 1, pp. 26–38). SCITEPRESS. https://doi.org/10.5220/0014655400004021

Results are presented in the order they appear in the paper, organized by section. For each result, an excerpt from the paper is given, followed by code to compute the result from the dataset provided. Example:

>This resulted in a data set of 12,485 LLM-enabled question sessions (6,847 exam question and 5,638 glossary comparison)...

```python
sessions = pd.concat( [ exam_writing_sessions, glossary_comparison_sessions ] )
print( f'{len( sessions )} sessions' )
```

Please refer to the paper for additional context.

---

In `glossary_comparison_sessions` and `fitb_sessions`, student answer attempts are classified using shorthand symbols. These symbols (`+`, `-`, `x`) are not used in the paper but correspond to the categories defined in the following table:

| Category    | Symbol | Description                                                                                             |
| ----------- | ------ | ------------------------------------------------------------------------------------------------------- |
| Correct     | `+`    | The response accurately addressed the key distinction between terms.                                    |
| Incorrect   | `-`    | The response did not sufficiently answer the question, despite appearing to be a genuine effort.        |
| Non-Genuine | `x`    | The response did not constitute a legitimate attempt (e.g., random characters, "idk", irrelevant text). |

`exam_writing_sessions` does not use these symbols — its `pattern` field records one `e` per answer attempt.

## Read dataset

In [1]:
import pandas as pd

LLM-enabled question sessions.

In [2]:
exam_writing_sessions = pd.read_parquet( 'exam_writing_sessions.parquet' )
glossary_comparison_sessions = pd.read_parquet( 'glossary_comparison_sessions.parquet' )

In [3]:
fitb_sessions = pd.read_parquet( 'fitb_sessions.parquet' )

In [4]:
exam_writing_sessions.head()

,question_id,student_id,course,textbook_id,question,pattern,answer,feedback,second_answer,second_feedback
0,0499745c5b03aec717885c833110d74965ce50efcb05c2...,2KQPBPGQKYRFKDFKYHA5,COM,9781544349848,"Write an exam question for the section ""Relati...",e,What are the 3 main parts of relational listen...,Your question does capture some important aspe...,None,None
1,0499745c5b03aec717885c833110d74965ce50efcb05c2...,2MQSTPV5RBCZNUTA4TTN,COM,9781544349848,"Write an exam question for the section ""Relati...",e,how is relational listening different across t...,"Your question, ""How is relational listening di...",None,None
2,0499745c5b03aec717885c833110d74965ce50efcb05c2...,2T6YX2UQS77MJ8VGNB7N,COM,9781544349848,"Write an exam question for the section ""Relati...",e,What is an example of relational listening?,"Your question, ""What is an example of relation...",None,None
3,0499745c5b03aec717885c833110d74965ce50efcb05c2...,2TWHRZM72RTYKHNRYVZF,COM,9781544349848,"Write an exam question for the section ""Relati...",e,Discuss the concept of relational listening an...,Your question does a good job of capturing the...,None,None
4,0499745c5b03aec717885c833110d74965ce50efcb05c2...,3AKC5SVKH3V33KUC6N3B,COM,9781544349848,"Write an exam question for the section ""Relati...",e,explain how relational listening shows the con...,"Your question, ""Explain how relational listeni...",None,None


In [5]:
glossary_comparison_sessions.head()

,question_id,student_id,course,textbook_id,question,pattern,first_attempt,second_attempt,answer,feedback,second_answer,second_feedback
0,041da273050f1811a3146414b755a91921f8c2f286315a...,2KQPBPGQKYRFKDFKYHA5,COM,9781544349848,Explain the difference between the term social...,+,+,None,Social roles are roles functioning to encourag...,Your explanation accurately captures the essen...,None,None
1,041da273050f1811a3146414b755a91921f8c2f286315a...,2MQSTPV5RBCZNUTA4TTN,COM,9781544349848,Explain the difference between the term social...,-,-,None,social roles are informal and task roles are f...,Your explanation touches on an aspect of the r...,None,None
2,041da273050f1811a3146414b755a91921f8c2f286315a...,2T6YX2UQS77MJ8VGNB7N,COM,9781544349848,Explain the difference between the term social...,+,+,None,Social roles are roles functioning to encurage...,Your explanation accurately captures the essen...,None,None
3,041da273050f1811a3146414b755a91921f8c2f286315a...,2TWHRZM72RTYKHNRYVZF,COM,9781544349848,Explain the difference between the term social...,+,+,None,Social roles aim to maintain group cohesion an...,Your explanation is accurate and well-articula...,None,None
4,041da273050f1811a3146414b755a91921f8c2f286315a...,3AKC5SVKH3V33KUC6N3B,COM,9781544349848,Explain the difference between the term social...,+,+,None,Social roles refer to behaviors and responsibi...,Your explanation is accurate and well-articula...,None,None


In [6]:
fitb_sessions.head()

,question_id,student_id,course,textbook_id,pattern,first_attempt,second_attempt
0,00309d69ce8b082b4630e979a4b7fccc33b03cf53c39ea...,3AKC5SVKH3V33KUC6N3B,CJ,9781071845226,xra+,x,+
1,00309d69ce8b082b4630e979a4b7fccc33b03cf53c39ea...,84AT2R3TAVDEJGPNHGM6,CJ,9781071845226,+,+,None
2,00309d69ce8b082b4630e979a4b7fccc33b03cf53c39ea...,8U2X4APT3CKCUXC4EBTX,CJ,9781071845226,+,+,None
3,00309d69ce8b082b4630e979a4b7fccc33b03cf53c39ea...,8U8N2HN4FBR6ED2V6PMR,CJ,9781071845226,-r+,-,+
4,00309d69ce8b082b4630e979a4b7fccc33b03cf53c39ea...,A8QHW6ZTFB6VBJZRVDSH,CJ,9781071845226,a+,+,None


## 2. Methods

### 2.4. Question Usage Data

>This resulted in a data set of 12,485 LLM-enabled question sessions (6,847 exam question and 5,638 glossary comparison), encompassing 13,889 interaction events, 101 questions, 246 students, and two different textbooks (Duck & McMahan, 2021; Mallicoat, 2023).

In [7]:
sessions = pd.concat( [ exam_writing_sessions, glossary_comparison_sessions ] )
print( f'{len( sessions )} LLM-enabled question sessions' )
print( f'{len( exam_writing_sessions )} exam question sessions' )
print( f'{len( glossary_comparison_sessions )} glossary comparison sessions' )
print( f'{sessions.pattern.apply( len ).sum()} interaction events' )
print( f'{sessions.question_id.nunique()} questions' )
print( f'{sessions.student_id.nunique()} students' )
print( f'{sessions.textbook_id.nunique()} textbooks' )

12485 LLM-enabled question sessions
6847 exam question sessions
5638 glossary comparison sessions
13889 interaction events
101 questions
246 students
2 textbooks


>For comparative purposes, data from the standard FITB questions (which constitute the majority of standard items) were retrieved for the same courses. The resulting FITB data included 52,995 student-question sessions.

In [8]:
print( f'{len( fitb_sessions )} FITB sessions' )

52995 FITB sessions


Per-course session counts. The two courses are CJ 4060 (Women, Gender, and Crime; 47 students) and COMST 1010 (Communication in Everyday Life; 205 students). Per-course analyses filter to sessions in the matching textbook.

In [9]:
CJ_TEXTBOOK  = '9781071845226'
COM_TEXTBOOK = '9781544349848'

for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    ew = exam_writing_sessions[         ( exam_writing_sessions.course         == course ) & ( exam_writing_sessions.textbook_id         == textbook ) ]
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    ecm = pd.concat( [ ew, gc ] )
    print( f'{course}: {ecm.student_id.nunique()} students / {len( ecm )} ECM sessions / {len( fb )} FITB sessions' )

CJ: 47 students / 959 ECM sessions / 7533 FITB sessions
COM: 200 students / 11526 ECM sessions / 45462 FITB sessions


## 3. Results and Discussion

### 3.1. Engagement

>**Table 1**<br/>
>Mean number of students answering by question type.
>
>|                     | CJ (N = 47) | COM (N = 205) |
>| ------------------- | ----------: | ------------: |
>| Exam Question       |        22.9 |         171.2 |
>| Glossary Comparison |        32.4 |         173.2 |
>| Standard FITB       |        37.1 |         174.9 |

In [10]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    ew = exam_writing_sessions[         ( exam_writing_sessions.course         == course ) & ( exam_writing_sessions.textbook_id         == textbook ) ]
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    rows.append( {
        'Course':           course,
        'Exam Question':    ew.groupby( 'question_id' ).student_id.nunique().mean().round( 1 ),
        'Glossary Comp.':   gc.groupby( 'question_id' ).student_id.nunique().mean().round( 1 ),
        'Standard FITB':    fb.groupby( 'question_id' ).student_id.nunique().mean().round( 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,Exam Question,Glossary Comp.,Standard FITB
Course,,,
CJ,22.9,32.4,37.1
COM,171.2,173.2,174.9


### 3.2. Difficulty

>As shown in Table 2, the glossary comparison questions exhibit a lower mean first-attempt accuracy than the standard FITB items in both CJ and COM. For CJ, glossary comparisons average a 69.8% accuracy rate, compared to 83.7% for standard FITB. In COM, the gap narrows somewhat (63.5% vs. 72.1%).
>
>**Table 2**<br/>
>Mean difficulty index by question type.
>
>|                     | CJ (N = 47) | COM (N = 205) |
>| ------------------- | ----------: | ------------: |
>| Glossary Comparison |        69.8 |          63.5 |
>| Standard FITB       |        83.7 |          72.1 |

In [11]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    rows.append( {
        'Course':           course,
        'Glossary Comp.':   ( ( gc.first_attempt == '+' ).mean() * 100 ).round( 1 ),
        'Standard FITB':    ( ( fb.first_attempt == '+' ).mean() * 100 ).round( 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,Glossary Comp.,Standard FITB
Course,,
CJ,69.5,83.7
COM,62.8,72.1


### 3.3. Persistence

>As shown in Table 3, persistence rates, defined as the percentage of students who eventually succeed after an incorrect first attempt, differ sharply between glossary comparison and FITB. This was not anticipated. In CJ, 14.1% of students persisted to a correct response for glossary comparisons, compared to 98.5% for FITB.
>
>**Table 3**<br/>
>Mean persistence by question type.
>
>|                     | CJ (N = 47) | COM (N = 205) |
>| ------------------- | ----------: | ------------: |
>| Glossary Comparison |        14.1 |          20.3 |
>| Standard FITB       |        98.5 |          88.6 |

In [12]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    # Among sessions with an incorrect first attempt, rate with eventual correct
    gc_persist = gc[ gc.first_attempt != '+' ].pattern.str.contains( r'\+' ).mean() * 100
    fb_persist = fb[ fb.first_attempt != '+' ].pattern.str.contains( r'\+' ).mean() * 100
    rows.append( {
        'Course':           course,
        'Glossary Comp.':   round( gc_persist, 1 ),
        'Standard FITB':    round( fb_persist, 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,Glossary Comp.,Standard FITB
Course,,
CJ,14.4,98.5
COM,20.6,88.6


>A straightforward follow-up analysis suggested by the above is to separate persistence for "reveal" and "non-reveal" FITB sessions. Some students do attempt to solve FITB items without clicking the reveal button, even if it is available. For these courses, the "reveal-less" FITB persistence rate is substantially lower, 20.0% for CJ and 13.9% for COM, which is much closer to the glossary comparison rate and actually lower than it for COM.

In the FITB session `pattern` field, `r` indicates a reveal-answer action.

In [13]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    fb = fitb_sessions[ ( fitb_sessions.course == course ) & ( fitb_sessions.textbook_id == textbook ) ]
    non_correct = fb[ fb.first_attempt != '+' ]
    no_reveal = non_correct[ ~non_correct.pattern.str.contains( 'r', case=False ) ]
    rows.append( {
        'Course':                 course,
        'FITB (all)':             round( non_correct.pattern.str.contains( r'\+' ).mean() * 100, 1 ),
        'FITB (reveal-less)':     round( no_reveal.pattern.str.contains( r'\+' ).sum() / len( non_correct ) * 100, 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,FITB (all),FITB (reveal-less)
Course,,
CJ,98.5,20.0
COM,88.6,13.9


>Although relatively few students revise their incorrect answers for glossary comparisons, those who do can demonstrate more thoughtful engagement (though some may simply copy the feedback verbatim). An example from CJ illustrates how a student fixed an initial error through feedback.

In [14]:
idx = 1854
row = glossary_comparison_sessions.loc[ idx ]
print( f'Prompt: {row.question}' )
print()
print( f'Attempt 1: {row.answer}' )
print()
print( f'Feedback: {row.feedback}' )
print()
print( f'Attempt 2: {row.second_answer}' )
print()
print( f'Feedback: {row.second_feedback}' )

Prompt: Explain the difference between the term Fear of victimization and Secondary victimization:

Attempt 1: Secondary victimization is when sexual assault victims have bad experiences with assistance, thus causing them more trauma. Fear of victimization is when someone is afraid to report something. 

Feedback: Your explanation needs some clarification. Secondary victimization refers to additional trauma experienced by victims, often due to negative responses from others or the system. Fear of victimization involves a heightened fear of crime, often influenced by media portrayals, and is not specifically about reporting. It's important to distinguish between the emotional impact of crime and the fear of potential crime.

Attempt 2: Secondary victimization is when a victim has additional trauma, often due to negative responses from others or the system. Fear of victimization involves a heightened fear of crime. 

Feedback: Your explanation captures the essence of both terms well. Sec

### 3.4. Non-Genuine Answers

>Table 4 presents the rate of non-genuine answer submissions: responses that appear to have no meaningful engagement with the question prompt. In CJ, glossary comparisons exhibit 8.0% non-genuine attempts, versus 2.8% for standard FITB. In COM, the gap is 14.5% vs. 8.5%.
>
>**Table 4**<br/>
>Percentage of non-genuine answers by question type.
>
>|                     | CJ (N = 47) | COM (N = 205) |
>| ------------------- | ----------: | ------------: |
>| Glossary Comparison |         8.0 |          14.5 |
>| Standard FITB       |         2.8 |           8.5 |

In [15]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    rows.append( {
        'Course':           course,
        'Glossary Comp. %': round( ( gc.first_attempt == 'x' ).mean() * 100, 1 ),
        'Standard FITB %':  round( ( fb.first_attempt == 'x' ).mean() * 100, 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,Glossary Comp. %,Standard FITB %
Course,,
CJ,8.3,2.8
COM,15.2,8.3


>As a follow-up analysis, it was checked whether more non-substantive answers occur on the first attempt than on subsequent attempts by the same student. If most non-genuine attempts happen immediately, it may indicate students are simply bypassing the task; if they occur later, students could be experiencing confusion or frustration. For both courses, the percentage of non-genuine answers dropped noticeably on the second attempt: 4.9% for glossary comparison and 0.6% for FITB in CJ, and 5.3% and 0.8%, respectively in COM.

In [16]:
rows = []
for course, textbook in [ ( 'CJ', CJ_TEXTBOOK ), ( 'COM', COM_TEXTBOOK ) ]:
    gc = glossary_comparison_sessions[  ( glossary_comparison_sessions.course  == course ) & ( glossary_comparison_sessions.textbook_id  == textbook ) ]
    fb = fitb_sessions[                 ( fitb_sessions.course                 == course ) & ( fitb_sessions.textbook_id                 == textbook ) ]
    # Second-attempt non-genuine: among sessions with a second attempt, rate of 'x'
    gc2 = gc[ gc.second_attempt.notna() ]
    fb2 = fb[ fb.second_attempt.notna() ]
    rows.append( {
        'Course':                    course,
        'Glossary 2nd attempt %':    round( ( gc2.second_attempt == 'x' ).mean() * 100, 1 ),
        'FITB 2nd attempt %':        round( ( fb2.second_attempt == 'x' ).mean() * 100, 1 ),
    } )
pd.DataFrame( rows ).set_index( 'Course' )

,Glossary 2nd attempt %,FITB 2nd attempt %
Course,,
CJ,4.9,0.6
COM,5.9,0.8
